<a href="https://colab.research.google.com/github/davisdefoe/machine-learning-lab-experiments/blob/main/lab%20exp-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import pandas as pd
import math
from collections import Counter
from pprint import pprint

# Step 1: Create dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],

    'Play': ['No', 'No', 'Yes', 'Yes',
             'Yes', 'No', 'Yes', 'No',
             'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
}

df = pd.DataFrame(data)

# Step 2: Calculate Entropy
def entropy(target):
    counts = Counter(target)
    total = len(target)

    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )

# Step 3: Calculate Information Gain
def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (
            len(subset) / len(df)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

# Step 4: Build ID3 Decision Tree
def id3(df, attributes, target):

    # If all samples belong to one class
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes remain
    if not attributes:
        return df[target].mode()[0]

    # Select attribute with highest Information Gain
    gains = {
        attr: information_gain(df, attr, target)
        for attr in attributes
    }

    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}

    remaining = [
        attr for attr in attributes
        if attr != best_attribute
    ]

    # Create branches
    for value in df[best_attribute].unique():

        subset = df[df[best_attribute] == value]

        tree[best_attribute][value] = id3(
            subset, remaining, target
        )

    return tree

# Step 5: Build the decision tree
attributes = list(df.columns[:-1])

print("Training Dataset:")
print(df)

print("\nInformation Gain:")
for attr in attributes:
    print(attr, ":", round(
        information_gain(df, attr, 'Play'), 4
    ))

tree = id3(df, attributes, 'Play')

print("\nDecision Tree:")
pprint(tree)

# Step 6: Classify a new sample
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    if value not in tree[attribute]:
        return "Unknown"

    return predict(tree[attribute][value], sample)

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(tree, new_sample)

print("\nNew Sample:")
print(new_sample)

print("\nClassification Result:", result)


Training Dataset:
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

Information Gain:
Outlook : 0.2467
Temperature : 0.0292
Humidity : 0.1518
Wind : 0.0481

Decision Tree:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}

New S